# Brain State Analysis with TRIBE v2

This notebook demonstrates how to derive **emotional** and **cognitive states** from TRIBE v2's predicted brain activity.

We use three approaches:
1. **ROI-Based Scoring** — Map predictions to named brain regions via the Destrieux atlas, compute normalized activation scores
2. **Differential Comparison** — Compare brain patterns from different stimulus categories (happy vs sad, etc.)
3. **Pattern Classification** — Match novel stimuli to reference brain signatures

> **How it works**: TRIBE v2 predicts fMRI-like brain activity on the fsaverage5 cortical mesh (~20k vertices). We group vertices into functional brain regions using the Destrieux atlas, z-score normalize, and apply sigmoid mapping to produce 0–1 scores per cognitive domain.

> **Inspired by**: The [Ad Brain Scorer](https://huggingface.co/spaces) HuggingFace space, which uses the same approach for ad effectiveness scoring.

## 1. Load the Model

Load the TRIBE v2 model and initialize the brain atlas.

In [ ]:
from tribev2.demo_utils import TribeModel
from tribev2.brain_states import (
    BrainAtlas, BrainStateProfiler, BrainStateClassifier,
    compute_normalized_scores, create_radar_chart, create_timeline_chart,
)
from tribev2.plotting import PlotBrain
from pathlib import Path
import numpy as np

CACHE_FOLDER = Path("./cache")

# Load model (reuses cache from previous runs)
model = TribeModel.from_pretrained(
    "facebook/tribev2",
    config_update={
        "data.text_feature.model_name": "alpindale/Llama-3.2-3B",
        "data.text_feature.device": "cpu",
    },
)

# Initialize atlas for region mapping
atlas = BrainAtlas()
plotter = PlotBrain(mesh="fsaverage5")

print(f"Model loaded. Atlas has {len(atlas.labels)} regions.")

## 2. Analyze a Single Stimulus

Let's predict brain activity for a piece of text and see which brain regions it activates.

In [ ]:
# Choose a text to analyze
text = """
She opened the letter and tears of joy streamed down her face.
After years of hard work, she had been accepted into her dream program.
Her parents hugged her tightly and told her how proud they were.
"""

# Save to file and get events
text_path = CACHE_FOLDER / "analysis_input.txt"
text_path.write_text(text.strip())

df = model.get_events_dataframe(text_path=str(text_path))
preds, segments = model.predict(events=df)
print(f"Predictions shape: {preds.shape}  (n_timesteps, n_vertices)")

### Compute Brain State Scores

Each region group gets a 0–1 score:
- **0.5** = average activation (baseline)
- **> 0.5** = above-average engagement
- **< 0.5** = below-average engagement

In [ ]:
# Compute normalized scores
result = compute_normalized_scores(preds, atlas)

print("=== Brain Region Activation Scores ===")
print(f"{'Region Group':<30} {'Score':>8}")
print("-" * 40)
for group, score in result['scores'].items():
    bar = '█' * int(score * 20) + '░' * (20 - int(score * 20))
    print(f"{group:<30} {score:>7.1%}  {bar}")

print()
print(f"Emotional Valence:    {result['valence']:+.3f}  (positive = happy)")
print(f"Learning Readiness:   {result['learning']:.3f}  (higher = more engaged)")
print(f"Attention Level:      {result['attention']:.3f}  (higher = more focused)")

### Radar Chart — Brain State Profile

In [ ]:
fig = create_radar_chart(result['scores'], title="Brain State Profile — Happy Text")
fig.show()

### Timeline — Activation Over Time

In [ ]:
# Show key groups over time
key_groups = ['prefrontal', 'reward_vmPFC', 'insula', 'temporal', 'visual', 'anterior_cingulate']
fig2 = create_timeline_chart(result['time_series'], selected_groups=key_groups)
fig2.show()

### Brain Surface Visualization

In [ ]:
# Visualize the mean predicted activation on the brain surface
fig3 = plotter.plot_brain(preds.mean(axis=0), cmap="fire", norm_percentile=99, vmin=0.6, alpha_cmap=(0, 0.2))
fig3

## 3. Compare Brain States: Happy vs Sad

The most powerful approach: run TRIBE v2 on stimuli with known emotional content, then compare the predicted brain patterns. The difference reveals exactly which brain regions distinguish happy from sad.

In [ ]:
# Initialize the profiler with our curated stimuli
profiler = BrainStateProfiler(
    model=model,
    stimuli_dir="./stimuli",
    cache_dir="./cache/brain_states",
)

# Build profiles for happy and sad (uses cache if available)
# NOTE: First run takes ~10-15 min (5 passages × 2-3 min each × 2 states)
profiles = profiler.build_profiles(states=["happy", "sad"], max_passages=2)
print(f"Built {len(profiles)} profiles: {list(profiles.keys())}")

In [ ]:
# Compare Happy vs Sad scores
for state_name, pattern in profiles.items():
    # Reshape to (1, 20484) for scoring
    state_preds = pattern.reshape(1, -1)
    state_result = compute_normalized_scores(state_preds, atlas)
    
    print(f"\n=== {state_name.upper()} ===")
    for group, score in state_result['scores'].items():
        bar = '█' * int(score * 20) + '░' * (20 - int(score * 20))
        print(f"  {group:<30} {score:>7.1%}  {bar}")
    print(f"  Valence: {state_result['valence']:+.3f}")

In [ ]:
# Compute and visualize the differential map
diff_map = profiler.differential_map("happy", "sad")
print(f"Differential map shape: {diff_map.shape}")
print(f"Max diff: {diff_map.max():.4f}, Min diff: {diff_map.min():.4f}")

# Visualize on brain surface (blue = sad > happy, red = happy > sad)
fig4 = plotter.plot_brain(diff_map, cmap="RdBu_r", norm_percentile=95, symmetric_cmap=True)
fig4

## 4. Classify a Novel Stimulus

Given the reference profiles, classify a new text by correlating its brain pattern to the references.

In [ ]:
# Create a classifier with the built profiles
classifier = BrainStateClassifier(profiles=profiles, atlas=atlas)

# Analyze the prediction we already have (from the happy text above)
analysis = classifier.analyze(preds)

print("=== Classification ===")
for state, corr in analysis['classification']:
    print(f"  {state}: r = {corr:.3f}")

print(f"\nEmotional Valence: {analysis['valence']:+.3f}")
print(f"Learning Readiness: {analysis['learning']:.3f}")
print(f"Attention Level: {analysis['attention']:.3f}")

## 5. Build More Profiles (Optional)

Extend the analysis to more emotional/cognitive states. Each additional state takes ~5-10 minutes to build.

In [ ]:
# Uncomment to build all 5 state profiles (takes ~50-75 min total)
# profiles = profiler.build_profiles(max_passages=3)
# print(f"Built {len(profiles)} profiles: {list(profiles.keys())}")

# Then create a full classifier:
# classifier = BrainStateClassifier(profiles=profiles, atlas=atlas)
# analysis = classifier.analyze(preds)
# for state, corr in analysis['classification']:
#     print(f"  {state}: r = {corr:.3f}")